# Week 4 — Statistical inference for strategy returns

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Estimate the standard error and confidence interval of a mean return.
- Build a bootstrap confidence interval for a mean return.
- Understand what a p-value means and how it is commonly misused.
- Demonstrate how 'testing many random strategies' manufactures false positives.

## Estimated study time

About 9–11 hours.

## Prerequisites

- Sampling uncertainty from Week 3
- Mean and standard deviation

## External resources

- [MIT OpenCourseWare 18.05 Introduction to Probability and Statistics](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/)
- [NTU OpenCourseWare Statistics I and Introductory Econometrics](https://ocw.aca.ntu.edu.tw/courses/112S103)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### Estimators, standard errors and confidence intervals

An **estimator** is a function of the data (for example the sample mean). It has **bias** (systematic error) and **variance** (it varies from sample to sample). The **standard error** of the sample mean is $s/\sqrt n$.

A **confidence interval** gives 'the range of parameter values compatible with the data'. The correct reading of a 95% confidence interval is: if we repeated the sampling many times, about 95% of the intervals would cover the true parameter.

### p-values and their misuse

A p-value is 'the probability of seeing a result this extreme or more extreme, **assuming the null hypothesis is true**'. It is **not** 'the probability that the strategy works'. The most dangerous misuse is **multiple testing**: test enough random strategies and a few will come out 'significant' on luck alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.math.probability import simulate_normal
from quant_math_roadmap.math.statistics import (
    bootstrap_mean_ci, confidence_interval_mean,
    false_discovery_demo, one_sample_ttest, standard_error_of_mean,
)
from quant_math_roadmap.finance.metrics import sharpe_ratio
import pandas as pd

### Standard error and confidence interval of a mean return

In [ ]:
returns = simulate_normal(mean=0.0004, std=0.012, size=252, seed=42)
se = standard_error_of_mean(returns)
lower, upper = confidence_interval_mean(returns, confidence=0.95)
print(f'Sample mean daily return = {returns.mean():.6f}')
print(f'Standard error = {se:.6f}')
print(f'95% confidence interval = [{lower:.6f}, {upper:.6f}]')
print('Note: the interval most likely covers 0 — we cannot rule out a true expected return of 0.')

### Bootstrap confidence interval

In [ ]:
boot_lower, boot_upper = bootstrap_mean_ci(
    returns, confidence=0.95, n_resamples=5000, seed=0)
print(f'bootstrap 95% confidence interval      = [{boot_lower:.6f}, {boot_upper:.6f}]')
print(f't-distribution 95% confidence interval = [{lower:.6f}, {upper:.6f}]')
print('The two methods give similar intervals; the bootstrap needs no normality assumption.')

### Block bootstrap: when returns are autocorrelated

The plain bootstrap treats every observation as independently resamplable — an implicit i.i.d. assumption. If returns are **autocorrelated** (momentum-style strategies often are), the plain bootstrap **understates** the uncertainty of the mean return. The **circular block bootstrap** resamples whole contiguous blocks instead, preserving the dependence structure within each block.

Below we use a highly autocorrelated AR(1) series to show the gap between the two methods.

In [ ]:
from quant_math_roadmap.math.statistics import block_bootstrap_mean_ci
from quant_math_roadmap.data import generate_ar1_series

# AR(1) with phi=0.9: the effective sample size is far smaller than the nominal one
persistent = generate_ar1_series(2000, phi=0.9, seed=7).to_numpy()
plain_ci = bootstrap_mean_ci(persistent, seed=0)
block_ci = block_bootstrap_mean_ci(persistent, block_size=50, seed=0)
print(f'plain bootstrap 95% CI width = {plain_ci[1] - plain_ci[0]:.4f}')
print(f'block bootstrap 95% CI width = {block_ci[1] - block_ci[0]:.4f}')
print('With autocorrelated data, the plain bootstrap gives an interval that is too narrow (overconfident).')

### Comparing two synthetic strategies

In [ ]:
strategy_a = simulate_normal(mean=0.0002, std=0.010, size=252, seed=1)
strategy_b = simulate_normal(mean=0.0007, std=0.018, size=252, seed=2)
for name, s in [('Strategy A', strategy_a), ('Strategy B', strategy_b)]:
    t = one_sample_ttest(s, popmean=0.0)
    ci = confidence_interval_mean(s)
    print(f'{name}: mean={s.mean():.6f}, p-value={t.p_value:.3f}, '
          f'95% CI=[{ci[0]:.6f}, {ci[1]:.6f}]')

Even when one strategy has the higher sample mean, its p-value can still be insignificant and its confidence interval can still cover 0. **A higher historical mean return does not equal a higher true expected return.**

### Multiple testing: a false-positive factory

Below we generate a large batch of **pure-noise** strategies (every true expected return is 0) and see how many get falsely flagged as 'significant' at $\alpha=0.05$.

In [ ]:
demo = false_discovery_demo(n_strategies=500, n_periods=252,
                            alpha=0.05, seed=0)
for k, v in demo.items():
    print(f'{k}: {v}')
print()
print('All 500 strategies are pure noise, so every "significant" result is a false positive.')

In [ ]:
# Visualization: the equity curve of the best noise strategy can look gorgeous too
rng = np.random.default_rng(0)
noise = rng.standard_normal((500, 252)) * 0.01
totals = (1 + noise).prod(axis=1) - 1
best = noise[int(np.argmax(totals))]
equity = (1 + pd.Series(best)).cumprod()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(equity.index, equity.values, label='The "best" of 500 noise strategies')
ax.axhline(1.0, linestyle='--', label='Starting capital')
ax.set_title('A beautiful equity curve — built on pure luck')
ax.set_xlabel('Trading day')
ax.set_ylabel('Equity (start = 1)')
ax.legend()
plt.show()

This curve was generated entirely from noise, yet it may look prettier than a strategy with genuine signal. **A beautiful equity curve cannot prove that a strategy works.**

### Turning 'multiple testing' into a number: PSR and the Deflated Sharpe Ratio

We just showed that 'test enough strategies and a few will look significant'. Bailey and López de Prado turned that warning into computable metrics:

- **PSR (Probabilistic Sharpe Ratio)**: the probability that the true Sharpe exceeds a benchmark, after accounting for sample length, skewness and kurtosis;
- **DSR (Deflated Sharpe Ratio)**: raises the benchmark from 0 to 'the expected Sharpe of the luckiest among N skill-less strategies' — the more strategies you tried, the higher the bar the winner has to clear.

In [ ]:
from quant_math_roadmap.finance.metrics import (
    deflated_sharpe_ratio, expected_max_sharpe, probabilistic_sharpe_ratio,
)

# Reuse the 500 pure-noise strategies from above: pick the 'champion' with the highest total return
best_returns = pd.Series(best)

# Per-period Sharpe estimates of each strategy; their cross-strategy std feeds the expected-max formula
per_period_sr = noise.mean(axis=1) / noise.std(axis=1, ddof=1)
sr_std = float(per_period_sr.std(ddof=1))

psr = probabilistic_sharpe_ratio(best_returns)
benchmark = expected_max_sharpe(500, sr_std=sr_std)
dsr = deflated_sharpe_ratio(best_returns, n_trials=500, sr_std=sr_std)
print(f'Champion strategy PSR (benchmark SR=0)        = {psr:.4f}  <- looks quite convincing')
print(f'Expected max SR across 500 trials             = {benchmark:.4f}')
print(f'Champion strategy DSR (selection effect removed) = {dsr:.4f}  <- the mask comes off')

The PSR looks high — but only because we **picked the luckiest strategy**. Once the benchmark honestly accounts for 'we tried 500 times', the DSR collapses: this strategy's 'outstanding' performance is indistinguishable from pure luck. **When reporting backtest results, you must also report how many configurations you tried in total.**

### A caution on risk-adjusted metrics

In [ ]:
sr = sharpe_ratio(pd.Series(returns), frequency='daily')
print(f'Annualized Sharpe ratio = {sr:.3f}')
print('Caution: the Sharpe ratio is an estimate with its own sampling error; it ignores skewness and fat tails;')
print('with a short sample, a backtest Sharpe of 2 can still be consistent with a true Sharpe of 0.')

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. Write the correct definition of a p-value in one sentence.
2. What is the correct interpretation of a '95% confidence interval'? What is the common incorrect one?
3. Why does 'testing many strategies and picking the best one' rob the p-value of its meaning?

### Applied exercises

In [ ]:
# Applied exercise 1: bootstrap strategy_a and compare the widths of the 90% and 99% confidence intervals.
ci90 = None  # TODO: bootstrap_mean_ci(strategy_a, confidence=0.90, seed=0)
ci99 = None  # TODO: bootstrap_mean_ci(strategy_a, confidence=0.99, seed=0)
if ci90 and ci99:
    print('90% width:', ci90[1] - ci90[0])
    print('99% width:', ci99[1] - ci99[0])

In [ ]:
# Applied exercise 2: change alpha to 0.01 in false_discovery_demo and
# watch how the number of false positives changes.
strict = None  # TODO: false_discovery_demo(n_strategies=500, alpha=0.01, seed=0)
if strict is not None:
    print(strict)

### Reflection question

1. You found a strategy with a great backtest among a large grid of parameter combinations. Before believing in it, what should you do about 'multiple testing' and 'out-of-sample validation'?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. What is the correct definition of a p-value?**
- A. The probability that the null hypothesis is true
- B. The probability of seeing a result this extreme or more extreme, given that the null hypothesis is true
- C. The probability that the strategy works
- D. The probability of making a Type I error

**Q2. What is the correct interpretation of a 95% confidence interval?**
- A. The parameter has a 95% probability of lying inside the interval
- B. Under repeated sampling, about 95% of the intervals would cover the true parameter
- C. 95% of the data falls inside the interval
- D. The prediction accuracy is 95%

**Q3. Testing 100 pure-noise strategies at significance level α=0.05, how many do we expect to be 'significant'?**
- A. 0
- B. 1
- C. 5
- D. 50

**Q4. What is the purpose of the block bootstrap relative to the plain bootstrap?**
- A. Faster computation
- B. Preserving the autocorrelation structure of the data
- C. Making the sample larger
- D. Reducing variance

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '304c8f72060c041b', 2: 'd62e543599a0a653', 3: 'af0be7d09fe40e70', 4: '83613d5a2b2977f8'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w4-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Reading a p-value as 'the probability that the strategy works'.**
- **Testing many strategies and reporting only the best one, without any multiple-testing correction.**
- **Mistaking statistical significance for economic significance (even a real effect can be eaten by costs).**
- **Drawing conclusions from a single backtest Sharpe ratio while ignoring its sampling error.**

## After this week, you should be able to

- [ ] Compute and interpret confidence intervals for a mean return (t-based and bootstrap).
- [ ] State the meaning of a p-value correctly.
- [ ] Demonstrate and explain the false positives created by multiple testing.
- [ ] Name at least three limitations of the Sharpe ratio.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.